# 02 · Data Preparation (CRISP-DM step 3)

**Goal:** make sure the provided train/val/test split is trustworthy, then decide how images
are prepared for the model.

1. **Exact duplicates:** identical files anywhere in the dataset.
2. **Near duplicates across splits:** if a test image is almost the same photo as a training
   image, the test score is inflated. This is called *data leakage*.
3. **Exclusion list:** files to leave out, saved to `02_data/interim/excluded_files.csv`.
   Notebooks 03–05 skip these files automatically. The raw images are never changed.
4. **Preprocessing and augmentation preview:** what the model will actually see.

In [ ]:
# --- SETUP: run this cell first -----------------------------------------
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Change this if you uploaded the project folder somewhere else in your Drive:
    PROJECT = Path("/content/drive/MyDrive/kapeklasipika")
    %pip install -q imagehash
else:
    # Running on your laptop: walk up from this notebook until we find config/config.yaml
    PROJECT = Path.cwd()
    while not (PROJECT / "config" / "config.yaml").exists():
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.config import load_config, use_local_copy
cfg = load_config()
print("Project folder:", PROJECT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.preprocessing import (find_exact_duplicates, find_near_duplicates,
                               confirm_near_duplicates, build_exclusions)
from src.data_loading import CLASS_NAMES

FIG = cfg["paths"]["figures"]; INTERIM = cfg["paths"]["interim"]
inv = pd.read_csv(INTERIM / "inventory.csv")   # created by notebook 01
inv = inv[inv["readable"]]

def show_pairs(pairs, title_col, save_name=None, n=8):
    """Draw flagged pairs side by side: left image | right image."""
    pairs = pairs.head(n)
    if not len(pairs):
        print("Nothing to show."); return
    fig, axes = plt.subplots(len(pairs), 2, figsize=(4.5, 2.2 * len(pairs)))
    axes = np.atleast_2d(axes)
    for (ax_a, ax_b), (_, r) in zip(axes, pairs.iterrows()):
        for ax, f, s, l in ((ax_a, r.file_a, r.split_a, r.label_a), (ax_b, r.file_b, r.split_b, r.label_b)):
            ax.imshow(Image.open(f).convert("RGB")); ax.axis("off")
            ax.set_title(f"{s} / {l}  ({title_col}={r[title_col]})", fontsize=8)
    plt.tight_layout()
    if save_name:
        plt.savefig(FIG / save_name, dpi=200)
    plt.show()

## 1. Exact duplicates\nFiles that are byte-for-byte identical (same MD5 checksum).

In [ ]:
exact = find_exact_duplicates(inv)
exact.to_csv(INTERIM / "exact_duplicates.csv", index=False)
groups = exact.groupby("md5").agg(n=("file_path", "size"),
                                  splits=("split", lambda s: "+".join(sorted(set(s)))),
                                  labels=("label", lambda s: "+".join(sorted(set(s)))))
print(f"{len(exact)} files in {len(groups)} groups of identical copies")
print("\nWhere the copies sit (split, label):")
groups.groupby(["splits", "labels"]).size().rename("groups")

**How to read this:** a group inside one split and one label (e.g. `test` / `peaberry`) is
just a duplicated file: harmless for training but it double-counts in the test score.
A group spanning two splits is leakage. A group with two labels means the same image was
labelled twice, so its label cannot be trusted.

## 2. Near duplicates across splits
**Step A: perceptual hash.** A 64-bit fingerprint of how an image looks; pairs whose
fingerprints differ by 3 bits or fewer are flagged. *This takes several minutes the first
time; afterwards the saved result is reused.*

In [ ]:
MAX_DISTANCE = 3
cache = INTERIM / "near_duplicates.csv"
if cache.exists():
    near = pd.read_csv(cache)
    print("Loaded saved hash results from", cache.name)
else:
    near = find_near_duplicates(inv, max_distance=MAX_DISTANCE)
    near.to_csv(cache, index=False)
print(f"{len(near)} pairs flagged by the hash across splits")
print("Flagged pairs with DIFFERENT labels:", (near["label_a"] != near["label_b"]).sum())

**Step B: pixel check.** The first run showed the hash over-flags this dataset: a dark bean
centred on a white background produces similar fingerprints even for different beans
(95 of 184 flagged pairs had different labels, and the pictures were clearly different beans).
So each flagged pair is also compared pixel by pixel: the average difference between the two
grayscale images on a 0–255 scale. The same photo stays within a few levels;
different beans differ by much more.

In [ ]:
MAX_PIXEL_DIFF = 3.0
checked = confirm_near_duplicates(near, max_pixel_diff=MAX_PIXEL_DIFF)
checked.to_csv(INTERIM / "near_duplicates_checked.csv", index=False)
confirmed = checked[checked["confirmed"]]
print(f"Confirmed duplicates: {len(confirmed)} of {len(checked)} flagged pairs (pixel diff <= {MAX_PIXEL_DIFF})")
if len(confirmed):
    display(confirmed.groupby(["split_a", "split_b"]).size().rename("pairs"))
    print("Confirmed pairs with different labels:", (confirmed["label_a"] != confirmed["label_b"]).sum())

ax = checked["pixel_diff"].plot(kind="hist", bins=40, figsize=(7, 3.5))
ax.axvline(MAX_PIXEL_DIFF, color="red", linestyle="--", label=f"threshold = {MAX_PIXEL_DIFF}")
ax.set_xlabel("Average pixel difference (0-255)"); ax.set_ylabel("Flagged pairs")
ax.set_title("Pixel difference of hash-flagged pairs"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "02_pixel_difference_hist.png", dpi=200); plt.show()

### Confirmed duplicates (should look like the same photo)

In [ ]:
show_pairs(confirmed, "pixel_diff", "02_confirmed_duplicates.png")

### Closest pairs that were NOT confirmed (should look like different beans)

In [ ]:
show_pairs(checked[~checked["confirmed"]], "pixel_diff", "02_rejected_pairs.png")

## 3. Exclusion list
Rules (also in `src/preprocessing.py`):
- Exact copies in the same split and label: keep one, exclude the rest.
- Duplicates across splits: exclude the copy in the later split (val or test), so the
  model is never evaluated on an image it trained on.
- Duplicates with conflicting labels: exclude every copy, because the label is unreliable.

In [ ]:
excluded = build_exclusions(exact, checked)
excluded.to_csv(cfg["paths"]["exclusions"], index=False)
excluded["split"] = excluded["relative_path"].str.split("/").str[0]
excluded["label"] = excluded["relative_path"].str.split("/").str[1]
print(f"{len(excluded)} files excluded -> {cfg['paths']['exclusions'].name}")
display(excluded.groupby(["reason", "split"]).size().rename("files"))

remaining = inv.assign(key=inv["file_path"].str.split("/").str[-3:].str.join("/"))
remaining = remaining[~remaining["key"].isin(set(excluded["relative_path"]))]
final = pd.crosstab(remaining["split"], remaining["label"]).reindex(["train", "val", "test"])[CLASS_NAMES]
final["total"] = final.sum(axis=1)
print("\nImages used from now on:")
final

## 4. Preprocessing and augmentation preview
**Preprocessing** (inside the model, so the phone app needs no extra code):
center-crop to a square → resize 256→224 px → scale pixels from 0–255 to −1…1.

**Augmentation** (training only): random flips, rotation up to 180°, zoom ±15%,
brightness ±20%, contrast ±20%. Each row below is one training image and five random
versions the model might see.

In [ ]:
import tensorflow as tf
from src.features import build_augmentation

aug = build_augmentation(seed=cfg["seed"])
examples = inv[inv["split"] == "train"].groupby("label").head(1)
fig, axes = plt.subplots(len(examples), 6, figsize=(12, 2.2 * len(examples)))
for row, (_, r) in zip(axes, examples.iterrows()):
    img = tf.image.resize(np.asarray(Image.open(r.file_path).convert("RGB"), dtype="float32"), (cfg["img_size"],) * 2)
    row[0].imshow(img.numpy().astype("uint8")); row[0].set_title(f"{r.label} (original)", fontsize=8)
    for ax in row[1:]:
        ax.imshow(np.clip(aug(img[None], training=True)[0].numpy(), 0, 255).astype("uint8"))
    for ax in row:
        ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "02_augmentation_examples.png", dpi=200); plt.show()